# Тестирование алгоритма на одном отзыве

In [ ]:
!pip install pyabsa

In [35]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import BertTokenizerFast

from pyabsa import AspectTermExtraction as ATEPC

import warnings
warnings.filterwarnings("ignore")

In [36]:
example_review = 'Хочу выразить благодарность всему коллективу 9 травмотолого-ортопедического отделения НПЦ Психоневрологии на Мичуринском проспекте, 74.'

### Общая тональность с помощью модели RuBERT for Sentiment Analysis и предобученной модели для Aspect Term Extraction and Sentiment Classification (ATESC) из библиотеки PyABSA

In [37]:
tokenizer = BertTokenizerFast.from_pretrained('blanchefort/rubert-base-cased-sentiment')
model = AutoModelForSequenceClassification.from_pretrained('blanchefort/rubert-base-cased-sentiment', return_dict=True)

@torch.no_grad()
def sentiment_predict(text):
    inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors='pt')
    outputs = model(**inputs)
    predicted = torch.nn.functional.softmax(outputs.logits, dim=1)
    predicted = torch.argmax(predicted, dim=1).numpy()
    return predicted


In [ ]:
aspect_extractor = ATEPC.AspectExtractor('multilingual')

In [39]:
review_sentiment = sentiment_predict(example_review)

if review_sentiment == 0:
    overall_sentiment = 'Нейтральная'
elif review_sentiment == 1:
    overall_sentiment = 'Позитивная'
else:
    overall_sentiment = 'Негативная'

review_atepc = aspect_extractor.predict(example_review, print_result=False, save_result=False)

result = {
    "текст": example_review,
    "общая_тональность": overall_sentiment,
    "аспекты": []
}

aspects = review_atepc['aspect']
sentiments = review_atepc['sentiment']
confidences = review_atepc['confidence']

for asp, sent, conf in zip(aspects, sentiments, confidences):
    result["аспекты"].append({
        "аспект": asp,
        "тональность": sent,
        "уверенность": round(conf, 4)
    })

print("Анализ отзыва:")
print(f"Текст: {result['текст']}")
print(f"Общая тональность: {result['общая_тональность']}\n")

if result["аспекты"]:
    print("Аспектный анализ:")
    for asp in result["аспекты"]:
        print(f"  • {asp['аспект']}: {asp['тональность']} (уверенность {asp['уверенность']})")
else:
    print("Аспекты не найдены.")

Анализ отзыва:
Текст: Хочу выразить благодарность всему коллективу 9 травмотолого-ортопедического отделения НПЦ Психоневрологии на Мичуринском проспекте, 74.
Общая тональность: Позитивная

Аспектный анализ:
  • коллективу: Positive (уверенность 0.9886)
